In [1]:
!pip install torch torchaudio transformers jiwer datasets librosa soundfile editdistance

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 26.7 MB/s eta 0:00:0000:0100:01


In [15]:
# ============================================================

# 1. УСТАНОВКА И ИМПОРТЫ
# ============================================================

import os
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchaudio
from torch.utils.data import Dataset, DataLoader
from pathlib import Path
import random
import editdistance
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')


# Проверка CUDA
def get_device():
    if torch.cuda.is_available():
        try:
            # Проверяем, работает ли CUDA
            x = torch.zeros(1).cuda()
            del x
            return torch.device('cuda')
        except Exception as e:
            print(f"CUDA available but not working: {e}")
            print("Falling back to CPU")
            return torch.device('cpu')
    else:
        print("CUDA not available, using CPU")
        return torch.device('cpu')

# Устанавливаем seeds для воспроизводимости
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(42)

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("CUDA version:", torch.version.cuda)
    print("GPU:", torch.cuda.get_device_name(0))


PyTorch version: 2.10.0+cu128
CUDA available: True
CUDA version: 12.8
GPU: Tesla T4


In [16]:
# ============================================================
# 2. КОНФИГУРАЦИЯ ПУТЕЙ
# ============================================================

DATA_PATH = "/kaggle/input/competitions/asr-2026-spoken-numbers-recognition-challenge"
WORKING_PATH = "/kaggle/working"

os.makedirs(WORKING_PATH, exist_ok=True)

print(f"\nData path: {DATA_PATH}")
print(f"Working path: {WORKING_PATH}")


Data path: /kaggle/input/competitions/asr-2026-spoken-numbers-recognition-challenge
Working path: /kaggle/working


In [17]:
# ============================================================
# 3. ЗАГРУЗКА ДАННЫХ
# ============================================================

train_df = pd.read_csv(os.path.join(DATA_PATH, "train.csv"))
dev_df = pd.read_csv(os.path.join(DATA_PATH, "dev.csv"))
test_df = pd.read_csv(os.path.join(DATA_PATH, "test.csv"))

print("\n" + "="*60)
print("DATASET INFO")
print("="*60)
print(f"Train samples: {len(train_df):,}")
print(f"Dev samples: {len(dev_df):,}")
print(f"Test samples: {len(test_df):,}")

print(f"\nUnique speakers in train: {train_df['spk_id'].nunique()}")
print(f"Unique speakers in dev: {dev_df['spk_id'].nunique()}")

print(f"\nNumber range in train: {train_df['transcription'].min():,} - {train_df['transcription'].max():,}")
print(f"Unique numbers in train: {train_df['transcription'].nunique():,}")

print("\nGender distribution:")
print(train_df['gender'].value_counts())

print("\nSample rate distribution:")
print(train_df['samplerate'].value_counts())

print("\nSample data:")
print(train_df.head())



DATASET INFO
Train samples: 12,553
Dev samples: 2,265
Test samples: 2,582

Unique speakers in train: 6
Unique speakers in dev: 10

Number range in train: 14 - 999,888
Unique numbers in train: 12,510

Gender distribution:
gender
female    7665
male      4888
Name: count, dtype: int64

Sample rate distribution:
samplerate
24000    10424
22050     2129
Name: count, dtype: int64

Sample data:
               filename  transcription spk_id  gender  ext  samplerate
0  train/0007c21c23.wav         139473  spk_E  female  wav       24000
1  train/000bee1b1d.wav         992597  spk_B    male  wav       24000
2  train/001a718902.wav         500869  spk_A  female  wav       22050
3  train/001e8e5565.wav         969908  spk_C    male  wav       22050
4  train/001ee5be6b.wav          80484  spk_E  female  wav       24000


In [18]:
# ============================================================
# 4. DATASET (БЫСТРАЯ ВЕРСИЯ С MEL-СПЕКТРОГРАММАМИ)
# ============================================================

class NumberAudioDataset(Dataset):
    """
    Датасет для распознавания цифр из аудио
    Использует Mel-спектрограммы и предсказывает цифры напрямую (0-9)
    """
    def __init__(self, df, data_path, split='train', is_train=True, 
                 target_sr=16000, max_length=6, n_mels=80):
        self.df = df.reset_index(drop=True)
        self.data_path = Path(data_path)
        self.split = split
        self.is_train = is_train
        self.target_sr = target_sr
        self.max_length = max_length
        self.max_samples = target_sr * max_length
        self.n_mels = n_mels
        
        # Mel-спектрограмма transform
        self.mel_transform = torchaudio.transforms.MelSpectrogram(
            sample_rate=self.target_sr,
            n_fft=400,
            hop_length=160,
            n_mels=self.n_mels,
            f_min=0,
            f_max=8000
        )
        
    def __len__(self):
        return len(self.df)
    
    def load_audio(self, filename):
        """Загрузка аудио файла"""
        filepath = self.data_path / filename
        
        try:
            waveform, sr = torchaudio.load(str(filepath))
            
            # Конвертируем в моно
            if waveform.shape[0] > 1:
                waveform = torch.mean(waveform, dim=0, keepdim=True)
            
            # Ресемплинг
            if sr != self.target_sr:
                resampler = torchaudio.transforms.Resample(sr, self.target_sr)
                waveform = resampler(waveform)
            
            return waveform.squeeze(0)
        except Exception as e:
            print(f"Error loading {filepath}: {e}")
            return torch.zeros(self.target_sr)
    
    def augment_waveform(self, waveform):
        """Аугментация аудио (только для train)"""
        if not self.is_train:
            return waveform
        
        # Добавление шума
        if random.random() < 0.5:
            noise_factor = random.uniform(0.001, 0.01)
            noise = torch.randn_like(waveform) * noise_factor
            waveform = waveform + noise
        
        # Изменение громкости
        if random.random() < 0.3:
            gain = random.uniform(0.7, 1.3)
            waveform = waveform * gain
        
        # Сдвиг по времени
        if random.random() < 0.3:
            shift_max = int(self.target_sr * 0.1)
            shift = random.randint(-shift_max, shift_max)
            waveform = torch.roll(waveform, shift)
        
        return waveform
    
    def pad_or_trim(self, waveform):
        """Паддинг или обрезка до фиксированной длины"""
        if waveform.shape[0] > self.max_samples:
            waveform = waveform[:self.max_samples]
        elif waveform.shape[0] < self.max_samples:
            padding = self.max_samples - waveform.shape[0]
            waveform = F.pad(waveform, (0, padding))
        return waveform
    
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        
        # Загружаем аудио
        waveform = self.load_audio(row['filename'])
        
        # Аугментация
        waveform = self.augment_waveform(waveform)
        
        # Паддинг/обрезка
        waveform = self.pad_or_trim(waveform)
        
        # Mel-спектрограмма
        mel_spec = self.mel_transform(waveform)
        mel_spec = torch.log(mel_spec + 1e-9)
        
        # Нормализация
        mel_spec = (mel_spec - mel_spec.mean()) / (mel_spec.std() + 1e-8)
        
        item = {
            'input': mel_spec,  # (n_mels, time)
            'filename': row['filename']
        }
        
        # Таргет: конвертируем число в последовательность цифр
        if 'transcription' in row and pd.notna(row['transcription']):
            digits = [int(d) for d in str(int(row['transcription']))]
            item['target'] = torch.tensor(digits, dtype=torch.long)
            item['target_length'] = len(digits)
        
        return item


def collate_fn(batch):
    """Собираем батч с паддингом"""
    inputs = torch.stack([item['input'] for item in batch])
    filenames = [item['filename'] for item in batch]
    
    result = {
        'input': inputs,
        'filename': filenames
    }
    
    if 'target' in batch[0]:
        max_target_len = max(item['target_length'] for item in batch)
        targets = torch.zeros(len(batch), max_target_len, dtype=torch.long)
        target_lengths = []
        
        for i, item in enumerate(batch):
            targets[i, :item['target_length']] = item['target']
            target_lengths.append(item['target_length'])
        
        result['target'] = targets
        result['target_length'] = torch.tensor(target_lengths, dtype=torch.long)
    
    return result


In [19]:
# ============================================================
# 5. МОДЕЛЬ (< 5M ПАРАМЕТРОВ)
# ============================================================

class CompactASR(nn.Module):
    """
    Компактная ASR модель для распознавания цифр
    num_classes = 11 (цифры 0-9 + blank для CTC)
    """
    def __init__(self, num_classes=11, hidden_dim=128, num_layers=2, n_mels=80):
        super().__init__()
        
        # CNN Encoder (эффективные свертки)
        self.conv1 = nn.Conv2d(1, 32, kernel_size=3, stride=2, padding=1)
        self.bn1 = nn.BatchNorm2d(32)
        
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, stride=2, padding=1)
        self.bn2 = nn.BatchNorm2d(64)
        
        self.conv3 = nn.Conv2d(64, 96, kernel_size=3, stride=2, padding=1)
        self.bn3 = nn.BatchNorm2d(96)
        
        # После 3 сверток с stride=2: n_mels=80 -> 10
        self.rnn_input_size = 96 * 10  # 960
        
        # Bidirectional LSTM
        self.rnn = nn.LSTM(
            input_size=self.rnn_input_size,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            bidirectional=True,
            dropout=0.1 if num_layers > 1 else 0
        )
        
        # Output layer
        self.fc = nn.Linear(hidden_dim * 2, num_classes)
        
        self.dropout = nn.Dropout(0.1)
        
    def forward(self, x):
        # x: (batch, n_mels, time)
        x = x.unsqueeze(1)  # (batch, 1, n_mels, time)
        
        # CNN layers
        x = F.relu(self.bn1(self.conv1(x)))
        x = F.relu(self.bn2(self.conv2(x)))
        x = F.relu(self.bn3(self.conv3(x)))
        
        # Reshape для RNN: (batch, time, channels*freq)
        batch, channels, freq, time = x.shape
        x = x.permute(0, 3, 1, 2)  # (batch, time, channels, freq)
        x = x.reshape(batch, time, channels * freq)
        
        # RNN
        x, _ = self.rnn(x)
        x = self.dropout(x)
        
        # Output
        x = self.fc(x)  # (batch, time, num_classes)
        
        return F.log_softmax(x, dim=-1)
    
    def count_parameters(self):
        return sum(p.numel() for p in self.parameters() if p.requires_grad)


# Проверка размера модели
print("\n" + "="*60)
print("MODEL INFO")
print("="*60)

model_test = CompactASR(num_classes=11, hidden_dim=128, num_layers=2)
num_params = model_test.count_parameters()
print(f"Model parameters: {num_params:,}")
print(f"Within limit (< 5M): {'✓ YES' if num_params < 5_000_000 else '✗ NO'}")
del model_test


MODEL INFO
Model parameters: 1,588,843
Within limit (< 5M): ✓ YES


In [20]:
# ============================================================
# 6. ОБУЧЕНИЕ
# ============================================================

def decode_predictions(log_probs):
    """
    CTC жадный декодинг: log_probs -> числа
    log_probs: (batch, time, num_classes)
    """
    predictions = []
    
    for log_prob in log_probs:
        # Greedy decoding
        pred = torch.argmax(log_prob, dim=-1).cpu().numpy()
        
        # CTC collapse: убираем повторы и blank (0)
        decoded_digits = []
        prev = None
        for p in pred:
            if p != 0 and p != prev:  # 0 = blank
                decoded_digits.append(p)
            prev = p
        
        # Конвертируем цифры в число
        if decoded_digits:
            try:
                number_str = ''.join([str(d) for d in decoded_digits])
                number = int(number_str)
                predictions.append(number)
            except:
                predictions.append(0)
        else:
            predictions.append(0)
    
    return predictions


def calculate_cer(predictions, targets):
    """Character Error Rate"""
    total_distance = 0
    total_length = 0
    
    for pred, target in zip(predictions, targets):
        pred_str = str(pred)
        target_str = str(int(target))
        
        distance = editdistance.eval(pred_str, target_str)
        total_distance += distance
        total_length += len(target_str)
    
    return total_distance / max(total_length, 1)


def train_epoch(model, dataloader, optimizer, criterion, device, scaler=None):
    """Одна эпоха обучения с mixed precision"""
    model.train()
    total_loss = 0
    num_batches = 0
    
    pbar = tqdm(dataloader, desc="Training")
    for batch in pbar:
        inputs = batch['input'].to(device)
        targets = batch['target'].to(device)
        target_lengths = batch['target_length'].to(device)
        
        optimizer.zero_grad()
        
        # Mixed precision training
        if scaler is not None:
            with torch.cuda.amp.autocast():
                log_probs = model(inputs)
                log_probs = log_probs.permute(1, 0, 2)  # (time, batch, classes)
                
                input_lengths = torch.full((inputs.size(0),), log_probs.size(0), 
                                          dtype=torch.long, device=device)
                
                loss = criterion(log_probs, targets, input_lengths, target_lengths)
            
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer)
            scaler.update()
        else:
            log_probs = model(inputs)
            log_probs = log_probs.permute(1, 0, 2)
            
            input_lengths = torch.full((inputs.size(0),), log_probs.size(0), 
                                      dtype=torch.long, device=device)
            
            loss = criterion(log_probs, targets, input_lengths, target_lengths)
            
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
        
        if not torch.isnan(loss) and not torch.isinf(loss):
            total_loss += loss.item()
            num_batches += 1
            pbar.set_postfix({'loss': f'{loss.item():.4f}'})
    
    return total_loss / max(num_batches, 1)


@torch.no_grad()
def evaluate(model, dataloader, device):
    """Оценка модели"""
    model.eval()
    all_predictions = []
    all_targets = []
    
    for batch in tqdm(dataloader, desc="Evaluating"):
        inputs = batch['input'].to(device)
        
        log_probs = model(inputs)
        predictions = decode_predictions(log_probs)
        all_predictions.extend(predictions)
        
        if 'target' in batch:
            for target, length in zip(batch['target'], batch['target_length']):
                target_digits = target[:length].cpu().numpy()
                target_num = int(''.join([str(d) for d in target_digits]))
                all_targets.append(target_num)
    
    if all_targets:
        cer = calculate_cer(all_predictions, all_targets)
        return cer, all_predictions
    else:
        return None, all_predictions


def train_model(train_df, dev_df, data_path, num_epochs=25, batch_size=32, lr=1e-3):
    """Основная функция обучения"""
    
    # Устройство (принудительно CPU если CUDA не работает)
    device = get_device()  # Используем CPU из-за проблем с CUDA
    print(f"\nUsing device: {device}")
    
    # Датасеты
    train_dataset = NumberAudioDataset(train_df, data_path, split='train', is_train=True)
    dev_dataset = NumberAudioDataset(dev_df, data_path, split='dev', is_train=False)
    
    train_loader = DataLoader(
        train_dataset, 
        batch_size=batch_size, 
        shuffle=True, 
        collate_fn=collate_fn, 
        num_workers=2,
        pin_memory=False
    )
    dev_loader = DataLoader(
        dev_dataset, 
        batch_size=batch_size, 
        shuffle=False, 
        collate_fn=collate_fn, 
        num_workers=2,
        pin_memory=False
    )
    
    # Модель
    model = CompactASR(num_classes=11, hidden_dim=128, num_layers=2).to(device)
    print(f"Model parameters: {model.count_parameters():,}")
    
    # Оптимизатор и планировщик
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-5)
    criterion = nn.CTCLoss(blank=0, zero_infinity=True)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='min', factor=0.5, patience=3
    )
    
    # Mixed precision scaler (только для CUDA)
    scaler = None  # torch.cuda.amp.GradScaler() if device.type == 'cuda' else None
    
    best_cer = float('inf')
    patience_counter = 0
    max_patience = 7
    
    print("\n" + "="*60)
    print("TRAINING START")
    print("="*60)
    
    for epoch in range(num_epochs):
        print(f"\nEpoch {epoch+1}/{num_epochs}")
        print("-" * 60)
        
        # Обучение
        train_loss = train_epoch(model, train_loader, optimizer, criterion, device, scaler)
        
        # Валидация
        dev_cer, _ = evaluate(model, dev_loader, device)
        
        # Обновление learning rate
        scheduler.step(dev_cer)
        
        print(f"Train Loss: {train_loss:.4f}")
        print(f"Dev CER: {dev_cer:.4f}")
        print(f"LR: {optimizer.param_groups[0]['lr']:.6f}")
        
        # Early stopping
        if dev_cer < best_cer:
            best_cer = dev_cer
            patience_counter = 0
            torch.save(model.state_dict(), os.path.join(WORKING_PATH, 'best_model.pt'))
            print(f"✓ Saved best model (CER: {best_cer:.4f})")
        else:
            patience_counter += 1
            print(f"No improvement ({patience_counter}/{max_patience})")
            
            if patience_counter >= max_patience:
                print("\nEarly stopping triggered!")
                break
    
    print("\n" + "="*60)
    print(f"TRAINING COMPLETE - Best CER: {best_cer:.4f}")
    print("="*60)
    
    return model

In [21]:
# ============================================================
# 7. СОЗДАНИЕ SUBMISSION
# ============================================================

def create_submission(model_path, test_df, data_path, output_file='submission.csv'):
    """Создание файла submission"""
    
    device = torch.device('cpu')
    print(f"\nCreating submission on {device}")
    
    # Загрузка модели
    model = CompactASR(num_classes=11, hidden_dim=128, num_layers=2).to(device)
    model.load_state_dict(torch.load(model_path, map_location=device))
    model.eval()
    
    # Датасет
    test_dataset = NumberAudioDataset(test_df, data_path, split='test', is_train=False)
    test_loader = DataLoader(
        test_dataset, 
        batch_size=32, 
        shuffle=False, 
        collate_fn=collate_fn, 
        num_workers=2
    )
    
    # Предсказания
    _, predictions = evaluate(model, test_loader, device)
    
    # Submission
    submission = pd.DataFrame({
        'filename': test_df['filename'],
        'transcription': predictions
    })
    
    submission.to_csv(output_file, index=False)
    
    print(f"\n✓ Submission saved to {output_file}")
    print(f"\nFirst 10 predictions:")
    print(submission.head(10))
    print(f"\nLast 10 predictions:")
    print(submission.tail(10))
    
    # Базовая статистика
    print(f"\nPrediction statistics:")
    print(f"  Min: {submission['transcription'].min():,}")
    print(f"  Max: {submission['transcription'].max():,}")
    print(f"  Mean: {submission['transcription'].mean():,.0f}")
    print(f"  Unique: {submission['transcription'].nunique():,}")
    
    return submission


In [22]:
# ============================================================
# 8. ЗАПУСК ОБУЧЕНИЯ И СОЗДАНИЕ SUBMISSION
# ============================================================

print("\n" + "="*60)
print("STARTING KAGGLE ASR COMPETITION PIPELINE")
print("="*60)

# Обучение модели
model = train_model(
    train_df=train_df,
    dev_df=dev_df,
    data_path=DATA_PATH,
    num_epochs=25,
    batch_size=32,
    lr=1e-3
)

# Создание submission
submission = create_submission(
    model_path=os.path.join(WORKING_PATH, 'best_model.pt'),
    test_df=test_df,
    data_path=DATA_PATH,
    output_file=os.path.join(WORKING_PATH, 'submission.csv')
)

print("\n" + "="*60)
print("✓ PIPELINE COMPLETE!")
print("="*60)
print(f"\nFiles created:")
print(f"  - {WORKING_PATH}/best_model.pt")
print(f"  - {WORKING_PATH}/submission.csv")


STARTING KAGGLE ASR COMPETITION PIPELINE

Using device: cuda
Model parameters: 1,588,843

TRAINING START

Epoch 1/25
------------------------------------------------------------


Evaluating: 100%|██████████| 71/71 [00:28<00:00,  2.47it/s]


Train Loss: 1.2642
Dev CER: 0.3399
LR: 0.001000
✓ Saved best model (CER: 0.3399)

Epoch 2/25
------------------------------------------------------------


Evaluating: 100%|██████████| 71/71 [00:18<00:00,  3.92it/s]


Train Loss: -0.2841
Dev CER: 0.2882
LR: 0.001000
✓ Saved best model (CER: 0.2882)

Epoch 3/25
------------------------------------------------------------


...

Train Loss: -0.3464
Dev CER: 0.2184
LR: 0.000500
No improvement (6/7)

Epoch 20/25
------------------------------------------------------------


Evaluating: 100%|██████████| 71/71 [00:17<00:00,  3.99it/s]


Train Loss: -0.3474
Dev CER: 0.2137
LR: 0.000500
No improvement (7/7)

Early stopping triggered!

TRAINING COMPLETE - Best CER: 0.2096

Creating submission on cpu


Evaluating: 100%|██████████| 81/81 [00:43<00:00,  1.85it/s]


✓ Submission saved to /kaggle/working/submission.csv

First 10 predictions:
              filename  transcription
0  test/d2440788a9.mp3         461694
1  test/e247dbf761.mp3          27729
2  test/071f4a5be7.mp3          79187
3  test/798bd15db7.mp3            648
4  test/58c0464ad5.mp3         861168
5  test/2042774c60.mp3          49918
6  test/d96a2defb1.mp3         113929
7  test/f86697a638.mp3           7483
8  test/d01bbbf7c1.mp3           5431
9  test/525ee1542f.mp3       18125129

Last 10 predictions:
                 filename  transcription
2572  test/d618a5d7f8.wav         276726
2573  test/dd7ec9f237.wav         937658
2574  test/ddee2b9fa9.wav          61641
2575  test/de4ec57525.wav          98983
2576  test/e01c523993.wav          69736
2577  test/e09cefb462.wav           3292
2578  test/e1ce45e316.wav          54936
2579  test/e234e323ff.wav         617361
2580  test/f09990b654.wav         614895
2581  test/f73c3590b5.wav         949593

Prediction statistics:
  Min: 1